In [6]:
import os

# Use .get() to safely check the key without throwing a KeyError
api_key = os.environ.get("OPENAI_API_KEY")

if not api_key or api_key in ["", "YOUR_API_KEY_HERE"]:
    raise Exception("API KEY MISSING OR INVALID. Please set your actual OpenAI API key.")
else:
    print("All good! Valid key found.")


from agents import Agent, Runner, function_tool
import pydantic
# Tell Pydantic not to fail when fields are missing during validation
pydantic.BaseModel.model_config['extra'] = 'ignore'

agent = Agent(
    name="MyAgent",
    model="gpt-5.4-mini",
    instructions="You are a helpful assistant.",
)


All good! Valid key found.


In [2]:
# Runner.run is a classmethod — you don't instantiate Runner.
result = await Runner.run(agent, "Hello, Where did the Minnesota Vikings play?", max_turns=10)

print(f"Last Agent: {result.last_agent.name}")
print("----")
print(result.final_output)

Last Agent: MyAgent
----
The Minnesota Vikings played at **Metropolitan Stadium** in **Bloomington, Minnesota** from **1961 to 1981**. They later moved to the **Metrodome**.


In [3]:
from tavily import TavilyClient

if os.environ["TAVILY_API_KEY"] is None:
  raise Exception("TAVILY_API_KEY MISSING")
else:
  print("all good with search")

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

all good with search


In [7]:
@function_tool
def tavily_search(query: str) -> str:
    """
    Perform a web search using Tavily and return a summarized result.
    """
    response = tavily_client.search(query,search_depth='advanced',max_results='5')
    results = response.get("results", [])
    return results or "No results found."

In [10]:

agent = Agent(
    name="Web Research Agent",
    model="gpt-5.4-mini",
    instructions="Use tavily_search when you need up-to-date info.",
    tools=[tavily_search],
)

result = await Runner.run(agent, "What is the timezone & date today?", max_turns=10)
print(f"Last Agent: {result.last_agent.name}")
print("----")
print(result.final_output)

Last Agent: Web Research Agent
----
Today’s date is **2026-07-16**.

I don’t have access to your device’s local timezone, so I can’t know your exact timezone unless you tell me your location or timezone name. If you want, I can help you figure it out.
